<a href="https://colab.research.google.com/github/Anthei0774/Advent-of-Code/blob/main/2023/Day_10_Pipe_Maze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
puzzle = '''-L|F7
7S-7|
L|7||
-L-J|
L|-JF'''

puzzle = '''7-F7-
.FJ|7
SJLL7
|F--J
LJ.LJ'''

puzzle = '''...........
.S-------7.
.|F-----7|.
.||.....||.
.||.....||.
.|L-7.F-J|.
.|..|.|..|.
.L--J.L--J.
...........'''

puzzle = '''..........
.S------7.
.|F----7|.
.||....||.
.||....||.
.|L-7F-J|.
.|..||..|.
.L--JL--J.
..........'''

puzzle = '''.F----7F7F7F7F-7....
.|F--7||||||||FJ....
.||.FJ||||||||L7....
FJL7L7LJLJ||LJ.L-7..
L--J.L7...LJS7F-7L7.
....F-J..F7FJ|L7L7L7
....L7.F7||L7|.L7L7|
.....|FJLJ|FJ|F7|.LJ
....FJL-7.||.||||...
....L---J.LJ.LJLJ...'''

puzzle = '''FF7FSF7F7F7F7F7F---7
L|LJ||||||||||||F--J
FL-7LJLJ||||||LJL-77
F--JF--7||LJLJ7F7FJ-
L---JF-JLJ.||-FJLJJ7
|F|F-JF---7F7-L7L|7|
|FFJF7L7F-JF7|JL---7
7-L-JL7||F7|L7F-7F7|
L.L7LFJ|||||FJL7||LJ
L7JLJL-JLJLJL--JLJ.L'''

with open('10.txt') as f: puzzle = f.read()

puzzle = puzzle.split('\n')
puzzle = [list(line) for line in puzzle]
puzzle

R = len(puzzle)
C = len(puzzle[0])

################################################################################
# PART 1

# find starting position
start = None
i = 0
while start is None:
    j = 0
    while j < C:
        if puzzle[i][j] == 'S':
            start = (i, j)
            break
        j += 1
    i += 1
    assert i != R
# print('Start:', start)

# create graph object based on symbols
graph = {}

for r in range(R):
    for c in range(C):
        s = puzzle[r][c]
        if s == '|': graph[(r, c)] = [(r - 1, c), (r + 1, c)]
        elif s == '-': graph[(r, c)] = [(r, c - 1), (r, c + 1)]
        elif s == 'L': graph[(r, c)] = [(r - 1, c), (r, c + 1)]
        elif s == 'J': graph[(r, c)] = [(r - 1, c), (r, c - 1)]
        elif s == '7': graph[(r, c)] = [(r, c - 1), (r + 1, c)]
        elif s == 'F': graph[(r, c)] = [(r, c + 1), (r + 1, c)]
        else:
            assert s in ['.', 'S']
            graph[(r, c)] = []

graph[start] = [n for n in graph if start in graph[n]].copy()
graph

# BFS go brrr...
d = 0
BFS_distances = {start : d}
to_check = graph[start].copy()

iter = 0
while True:

    d += 1
    next_iter = []

    while to_check != []:
        checking = to_check.pop(0)
        BFS_distances[checking] = d
        # print('Checking:', checking)

        neighbours = graph[checking]
        neighbours = [n for n in neighbours if n not in BFS_distances and n not in to_check and n != checking]
        assert len(neighbours) <= 1
        # print('Neighbours:', neighbours)

        if len(neighbours) == 1: next_iter.append(neighbours[0])

    if next_iter == []:
        break
    else:
        to_check = next_iter

max(BFS_distances.values())

###############################################################################
# PART 2

# remove non-loop pipes
for r in range(R):
    for c in range(C):
        if (r, c) not in BFS_distances:
            puzzle[r][c] = '.'
            del graph[(r, c)]

graph = {node: {'neighbours': graph[node], 'direction': None} for node in graph}
graph

# https://en.wikipedia.org/wiki/Point_in_polygon
# top left item is 'up'
top_left = list(graph)[0]
r, c = top_left
assert puzzle[r][c] in ['S', 'F']

# give direction to each node
prev = (r, c)
next = (r, c + 1)
graph[prev]['direction'] = (-1, 1)
while next != top_left:

    # find next
    neighbours = [n for n in graph[next]['neighbours'] if n != prev]
    assert len(neighbours) == 1
    nextnext = neighbours[0]
    graph[next]['direction'] = (nextnext[0] - prev[0], nextnext[1] - prev[1])

    # step
    prev = next
    next = nextnext

graph

# Point-in-Polygon: PiP go brrrrr
inside = False
inside_area = 0
outside_area = 0
for r in range(R):
    for c in range(C):

        if (r, c) in graph:
            if graph[(r, c)]['direction'][0] < 0: inside = True
            elif graph[(r, c)]['direction'][0] > 0: inside = False
            else: pass
        else:
            inside_area += inside
            outside_area += (not inside)

assert inside_area + outside_area == R * C - len(graph)
inside_area